In [1]:
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader


In [2]:
neo4j_uri = "bolt://localhost:7687"
neo4j_username = "neo4j"
neo4j_password = "neo4jpassword"

In [ ]:
# driver = GraphDatabase.driver(uri, auth=("neo4j", "neo4jpassword"))

# with driver.session() as session:
#     result = session.run("RETURN 1 AS n")
#     for record in result:
#         print(record["n"])


In [14]:
import uuid
import json
from typing import List, Dict, Any

from neo4j import GraphDatabase

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document



# ------------------------------
# 1. Neo4j setup
# ------------------------------

neo4j_uri = "bolt://localhost:7687"
neo4j_username = "neo4j"
neo4j_password = "neo4jpassword"

driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)



# ------------------------------
# 2. LLM for KG extraction
# ------------------------------

ex_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    model_kwargs={"response_format": {"type": "json_object"}}
)

# Graph transformer using your ontology
graph_transformer = LLMGraphTransformer(
    llm=ex_llm,
    allowed_nodes=[
        "Dataset", "DataType", "SpatialDomain", "DataAttribute",
        "VisualizationTechnique", "RenderingMethod", "FeatureExtractionMethod",
        "VisualizationTask", "UserGoal", "Feature", "Variable", "Phenomenon",
        "SoftwareSystem", "Algorithm", "Library"
    ],
    allowed_relationships=[
        "HAS_DATATYPE", "HAS_ATTRIBUTE", "DEFINED_ON", "CONTAINS_FEATURE",
        "USES_METHOD", "EXTRACTS_FEATURE", "SUPPORTS_TASK",
        "VISUALIZES_VARIABLE", "VISUALIZES_PHENOMENON",
        "IMPLEMENTED_IN", "RUNS_IN", "SUPPORTED_BY",
        "RELATES_TO", "CAUSES", "INDICATES"
    ],
    strict_mode=True
)



# ------------------------------
# 3. Embeddings
# ------------------------------

embedder = OpenAIEmbeddings(model="text-embedding-3-small")



# ------------------------------
# 4. Text splitter
# ------------------------------

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150
)



# ------------------------------
# 5. Neo4j write function
# ------------------------------

def upsert_graph(session, graph_doc, chunk_id, chunk_text, source, page, embedding):

    # Create Chunk node
    session.run("""
        MERGE (c:Chunk {id: $chunk_id})
        SET c.text = $text,
            c.source = $source,
            c.page = $page,
            c.embedding = $embedding
    """, {
        "chunk_id": chunk_id,
        "text": chunk_text,
        "source": source,
        "page": page,
        "embedding": embedding,
    })

    # Create nodes
    for node in graph_doc.nodes:
        session.run(
            f"""
            MERGE (n:{node.type} {{id: $id}})
            SET n.name = $name
            """,
            {
                "id": node.id,
                "name": node.properties.get("name", node.id)
            }
        )

        # Connect Chunk → Node
        session.run("""
            MATCH (c:Chunk {id: $cid})
            MATCH (n {id: $nid})
            MERGE (c)-[:MENTIONS]->(n)
        """, {"cid": chunk_id, "nid": node.id})

    # Create relationships
    for rel in graph_doc.relationships:
        session.run(
            f"""
            MATCH (s {{id: $src}})
            MATCH (t {{id: $tgt}})
            MERGE (s)-[:{rel.type}]->(t)
            """,
            {"src": rel.source.id, "tgt": rel.target.id}
        )



# ------------------------------
# 6. PDF processing
# ------------------------------

def process_pdf(pdf_path):
    print(f"\n=== Processing {pdf_path} ===")

    docs = PyPDFLoader(pdf_path).load()
    chunks = text_splitter.split_documents(docs)

    with driver.session() as session:
        for i, doc in enumerate(chunks):

            chunk_text = doc.page_content
            page = doc.metadata.get("page", None)

            # Embedding
            embedding = embedder.embed_query(chunk_text)

            # KG extraction (YOUR correct API)
            graph_doc = graph_transformer.process_response(doc)

            # Store everything in Neo4j
            chunk_id = str(uuid.uuid4())

            upsert_graph(
                session,
                graph_doc,
                chunk_id,
                chunk_text,
                pdf_path,
                page,
                embedding
            )

            print(f"✓ Processed chunk {i + 1}/{len(chunks)}")


# ------------------------------
# 7. Run pipeline
# ------------------------------

pdf_file_paths = [
    "resources/KaufmanVolumeVisualization.pdf",
    "resources/Volume Visualization and Volume Rendering Techniques.pdf",
]

for pdf_path in pdf_file_paths:
    process_pdf(pdf_path)

print("\n=== DONE: All PDFs processed ===")



=== Processing resources/KaufmanVolumeVisualization.pdf ===
✓ Processed chunk 1/156
✓ Processed chunk 2/156
✓ Processed chunk 3/156
✓ Processed chunk 4/156
✓ Processed chunk 5/156
✓ Processed chunk 6/156
✓ Processed chunk 7/156
✓ Processed chunk 8/156
✓ Processed chunk 9/156
✓ Processed chunk 10/156
✓ Processed chunk 11/156
✓ Processed chunk 12/156
✓ Processed chunk 13/156
✓ Processed chunk 14/156
✓ Processed chunk 15/156
✓ Processed chunk 16/156
✓ Processed chunk 17/156
✓ Processed chunk 18/156
✓ Processed chunk 19/156
✓ Processed chunk 20/156
✓ Processed chunk 21/156
✓ Processed chunk 22/156
✓ Processed chunk 23/156
✓ Processed chunk 24/156
✓ Processed chunk 25/156
✓ Processed chunk 26/156
✓ Processed chunk 27/156
✓ Processed chunk 28/156
✓ Processed chunk 29/156
✓ Processed chunk 30/156
✓ Processed chunk 31/156
✓ Processed chunk 32/156
✓ Processed chunk 33/156
✓ Processed chunk 34/156
✓ Processed chunk 35/156
✓ Processed chunk 36/156
✓ Processed chunk 37/156
✓ Processed chunk 38/15